# Prepare Modeling Dataset

Assembles the final modeling dataset by merging the baseline demographic/utilisation features with the twelve-category comorbidity grouping (see `02_comorbidity_grouping.ipynb`), then builds the horizon labels and the train/test split.

Train/test split: `train_test_split` on `df.index`, stratified on `incident_cvd`, `random_state=42`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

pd.set_option('display.max_columns', None)

## 1. Load baseline features + merge comorbidity grouping features

In [2]:
df = pd.read_csv('baseline_features.csv')
grouping = pd.read_csv('comorbidity_grouping_features.csv')
df = df.merge(grouping, on='person_id', how='left', validate='one_to_one')

print(f'{len(df):,} patients, {df.shape[1]} columns')
print(f'Median follow-up: {df["years_followup"].median():.2f} years')
df.head()

3,050 patients, 45 columns
Median follow-up: 2.89 years


,person_id,sex,age,postcode,incident_cvd,years_followup,comorbid_Alcohol,comorbid_Anemia,comorbid_Arthritis,comorbid_Asthma,comorbid_BackPain,comorbid_BloodLoss,comorbid_CKD,comorbid_COPD,comorbid_Cancer,comorbid_Coagulopathy,comorbid_Drugs,comorbid_FluidsLytes,comorbid_Gout,comorbid_Hypothyroid,comorbid_Liver,comorbid_MHC,comorbid_NeuroOther,comorbid_Obesity,comorbid_Osteo,comorbid_PUD,comorbid_Paralysis,comorbid_WeightLoss,baseline_n_diagnoses,baseline_n_procedures,baseline_n_claims,baseline_n_episodes,baseline_history_days,cat_Renal,cat_FluidElectrolyte,cat_Metabolic,cat_Respiratory,cat_MentalHealthSubstance,cat_Musculoskeletal,cat_Hematologic,cat_Hepatic,cat_Neurologic,cat_Oncologic,cat_EndocrineOther,cat_ConstitutionalFrailty
0,438,F,95,5044.0,True,0.00000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,0,2,2,4430,0,0,0,0,0,0,0,0,0,0,0,0
1,683,M,106,2001.0,False,16.15332,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,686,F,102,2073.0,False,18.53525,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,12,4,3,3,0,0,0,0,1,0,0,0,0,0,1,0,0
3,939,F,102,2250.0,True,0.00000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
4,951,M,101,4165.0,False,18.53525,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,17,10,5,5,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2. Horizon labels (identical to baseline)

In [3]:
HORIZONS = [1, 2, 3, 4, 5]

for h in HORIZONS:
    has_event_by_h = df['incident_cvd'] & (df['years_followup'] <= h)
    observed_to_h = df['years_followup'] >= h
    df[f'label_{h}y'] = np.where(has_event_by_h, 1, np.where(observed_to_h, 0, np.nan))
    df[f'eligible_{h}y'] = has_event_by_h | observed_to_h

for h in HORIZONS:
    n_elig = int(df[f'eligible_{h}y'].sum())
    n_event = int((df[f'label_{h}y'] == 1).sum())
    print(f'{h}y horizon: {n_elig:,}/{len(df):,} eligible ({100*n_elig/len(df):.1f}%), '
          f'{n_event:,} events ({100*n_event/n_elig:.1f}% of eligible)')

1y horizon: 2,686/3,050 eligible (88.1%), 261 events (9.7% of eligible)
2y horizon: 2,289/3,050 eligible (75.0%), 328 events (14.3% of eligible)
3y horizon: 1,845/3,050 eligible (60.5%), 380 events (20.6% of eligible)
4y horizon: 1,459/3,050 eligible (47.8%), 411 events (28.2% of eligible)
5y horizon: 956/3,050 eligible (31.3%), 428 events (44.8% of eligible)


## 3. Feature columns

In [4]:
COMORBID_FEATURE_COLS = ['cat_Renal', 'cat_FluidElectrolyte', 'cat_Metabolic', 'cat_Respiratory',
                          'cat_MentalHealthSubstance', 'cat_Musculoskeletal', 'cat_Hematologic', 'cat_Hepatic',
                          'cat_Neurologic', 'cat_Oncologic', 'cat_EndocrineOther', 'cat_ConstitutionalFrailty']

UTILISATION_COLS = ['baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                     'baseline_n_episodes', 'baseline_history_days']
DEMO_COLS = ['sex', 'age']

FEATURE_COLS = DEMO_COLS + COMORBID_FEATURE_COLS + UTILISATION_COLS
print(f'{len(FEATURE_COLS)} features:')
print(FEATURE_COLS)

19 features:
['sex', 'age', 'cat_Renal', 'cat_FluidElectrolyte', 'cat_Metabolic', 'cat_Respiratory', 'cat_MentalHealthSubstance', 'cat_Musculoskeletal', 'cat_Hematologic', 'cat_Hepatic', 'cat_Neurologic', 'cat_Oncologic', 'cat_EndocrineOther', 'cat_ConstitutionalFrailty', 'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims', 'baseline_n_episodes', 'baseline_history_days']


## 4. Train/test split (identical procedure to baseline)

In [5]:
train_idx, test_idx = train_test_split(
    df.index, test_size=0.20, stratify=df['incident_cvd'], random_state=RANDOM_STATE
)
df['split'] = 'train'
df.loc[test_idx, 'split'] = 'test'

print(df['split'].value_counts())
print()
print('Event rate by split:')
print(df.groupby('split')['incident_cvd'].mean())

split
train    2440
test      610
Name: count, dtype: int64

Event rate by split:
split
test     0.154098
train    0.154918
Name: incident_cvd, dtype: float64


## 5. Drop comorbidity-derived columns not used by this variant, then save

`04_train_classifiers.ipynb` auto-detects the comorbidity feature columns generically (everything in the saved file that isn't a known demographic/utilisation/outcome/label column)

In [6]:
ALL_COMORBID_DERIVED_COLS = (
    [c for c in df.columns if c.startswith('comorbid_')]
    + ['comorbidity_count']
    + [c for c in df.columns if c.startswith('cat_')]
    + [c for c in df.columns if c.startswith('empcluster_')]
)
cols_to_drop = [c for c in ALL_COMORBID_DERIVED_COLS if c not in COMORBID_FEATURE_COLS]
cols_before = set(df.columns)
df = df.drop(columns=cols_to_drop, errors='ignore')
n_actually_dropped = len(cols_before - set(df.columns))
print(f'Dropped {n_actually_dropped} comorbidity-derived columns not used by this variant, kept {len(COMORBID_FEATURE_COLS)}.')

OUT_FILE = 'modeling_dataset.csv'
df.to_csv(OUT_FILE, index=False)
print(f'Saved {len(df):,} rows x {df.shape[1]} columns to {OUT_FILE}')

Dropped 22 comorbidity-derived columns not used by this variant, kept 12.
Saved 3,050 rows x 34 columns to modeling_dataset.csv
